# Transformation, Scaling, Binning, Skew Fixing and Sampling

**Objective:** Demonstrate transformation, feature scaling, binning, skew correction, and sampling.

**Dataset:** Synthetic customer value dataset

This notebook is Colab-ready and saves tables, metrics, and visual outputs under
`results/`. Public datasets or compact sample datasets are used so the workflow
remains reproducible.


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(42)


In [ ]:
df = pd.DataFrame(
    {
        "customer_value": rng.lognormal(mean=3.2, sigma=0.8, size=700),
        "frequency": rng.poisson(5, 700),
        "target": rng.choice([0, 1], 700, p=[0.65, 0.35]),
    }
)
df["log_value"] = np.log1p(df["customer_value"])
scaler = StandardScaler()
df["scaled_log_value"] = scaler.fit_transform(df[["log_value"]])
df["value_bin"] = pd.qcut(df["customer_value"], q=5, labels=["very_low", "low", "medium", "high", "very_high"])
train, test = train_test_split(df, test_size=0.3, random_state=42, stratify=df["target"])

summary = pd.DataFrame(
    {
        "metric": ["raw_skew", "log_skew", "train_rows", "test_rows"],
        "value": [df["customer_value"].skew(), df["log_value"].skew(), len(train), len(test)],
    }
)
summary.to_csv(RESULTS_DIR / "transformation_summary.csv", index=False)
df.to_csv(RESULTS_DIR / "transformed_dataset.csv", index=False)
display(summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df["customer_value"], bins=30, ax=axes[0], color="coral")
axes[0].set_title("Before: Skewed Customer Value")
sns.histplot(df["scaled_log_value"], bins=30, ax=axes[1], color="teal")
axes[1].set_title("After: Log Transform + Scaling")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "transformation_dashboard.png", dpi=180)
plt.show()
